# 🎗️ Breast Cancer Detection
### EfficientNetB0 — Ultrasound Image Classification

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile, os
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

zip_path = '/content/drive/MyDrive/Datasets/Breast_cancer_detection.zip'
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/breast')
print('Extracted!')

# Check folder structure
for root, dirs, files in os.walk('/content/breast'):
    if dirs:
        print(root, '->', dirs)

In [ ]:
# Find dataset folder with classes
data_dir = None
for root, dirs, files in os.walk('/content/breast'):
    # Look for folder containing class subfolders
    dl = [d.lower() for d in dirs]
    if any(x in dl for x in ['benign', 'malignant', 'normal']):
        data_dir = root
        break

print('Data dir:', data_dir)
print('Classes:', os.listdir(data_dir))

In [ ]:
IMG_SIZE = 224
BATCH    = 32

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.15,
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1,
    validation_split=0.2
)

train_data = train_gen.flow_from_directory(
    data_dir, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH, class_mode='categorical',
    subset='training', seed=42)

val_data = train_gen.flow_from_directory(
    data_dir, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH, class_mode='categorical',
    subset='validation', seed=42)

print('Classes:', train_data.class_indices)
print('Train:', train_data.samples, '| Val:', val_data.samples)

In [ ]:
num_classes = len(train_data.class_indices)
print(f'Number of classes: {num_classes}')

base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False

inp = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x   = base_model(inp, training=False)
x   = layers.GlobalAveragePooling2D()(x)
x   = layers.BatchNormalization()(x)
x   = layers.Dense(256, activation='relu')(x)
x   = layers.Dropout(0.4)(x)

if num_classes == 2:
    out  = layers.Dense(1, activation='sigmoid')(x)
    loss = 'binary_crossentropy'
    mode = 'binary'
else:
    out  = layers.Dense(num_classes, activation='softmax')(x)
    loss = 'categorical_crossentropy'
    mode = 'categorical'

model = Model(inp, out)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=loss,
    metrics=['accuracy']
)
model.summary()

In [ ]:
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor='val_accuracy'),
    ReduceLROnPlateau(factor=0.3, patience=3, monitor='val_loss')
]

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=callbacks
)
print(f'Phase 1 Best: {max(history.history["val_accuracy"])*100:.2f}%')

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss=loss,
    metrics=['accuracy']
)

history2 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=callbacks
)
print(f'Phase 2 Best: {max(history2.history["val_accuracy"])*100:.2f}%')

In [ ]:
import json, numpy as np

save_dir = '/content/drive/MyDrive/ml_models'
os.makedirs(save_dir, exist_ok=True)
model.save(f'{save_dir}/breast_model.h5')

with open(f'{save_dir}/breast_classes.json', 'w') as f:
    json.dump(train_data.class_indices, f)

print('✅ breast_model.h5 saved!')
print('✅ breast_classes.json saved!')
print('Classes:', train_data.class_indices)